In [1]:
from qiskit import QuantumRegister, ClassicalRegister,QuantumCircuit
import numpy as np
import random
from qiskit.quantum_info import Pauli, SparsePauliOp, Statevector,Operator
from qiskit.visualization import circuit_drawer, plot_histogram
import matplotlib.pyplot as plt
from qiskit_aer import AerSimulator
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.fake_provider import FakeFez
import re
from qiskit.primitives import Estimator

In [2]:
# Comment the next two command lines if no acces to real quantum hardware is granted
# service = QiskitRuntimeService(channel ="ibm_quantum_platform", token= 'API'
#                               , instance = 'CRN')
# backend = service.least_busy(simulator=False, operational=True)

# backend = FakeFez()# uncomment this command line if needed to run with fake backend
backend = AerSimulator()# uncomment this command line if needed to run with free noise
pm = generate_preset_pass_manager(optimization_level = 1,
                             backend = backend,
                             seed_transpiler = 24578)

In [3]:
class QKD_Networks:
    def __init__(self, nq, N, backend):

        if isinstance(nq, int) == False:
            raise TypeError("The number of Users must be an integer greater than or equal to 2")
        else:  
            self.nq = nq
            self.N = N
            self.QR = QuantumRegister(nq)
            self.CR = ClassicalRegister(nq)
            self.QR_e = QuantumRegister(nq)
            self.CR_e = ClassicalRegister(nq) 

    def basis(self,nq,N):
        random.seed(17555)
        basis_1 = ['X','V','Y','R']
        basis_k = ['X','Y']
        basis_choices = [[random.choice(basis_1) for _ in range(N)]]+[[random.choice(basis_k) 
                                                    for _ in range(N)] for _ in range(nq-1) ]
        return basis_choices
    # basis_choices = basis(nq,N)

    def N_particles_state(self,nq):
        qc = QuantumCircuit(self.QR, self.CR, self.CR_e )
        qc.h(self.QR[0])
        for k in range(1,nq):
            qc.cx(self.QR[0],self.QR[k])
        return qc

    
    def legitimate_users_mes_circs(self,nq,N):
        #  Legitimate_parties_1's qubit onto the X basis
        Legitimate_parties_1_X = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_X.x(self.QR[0])
        Legitimate_parties_1_X.measure(self.QR[0],self.CR[0])
        
        # Legitimate_parties_1's qubit onto the Y basis 
        Legitimate_parties_1_Y = QuantumCircuit(self.QR,self.CR,self.CR_e)
        # Legitimate_parties_1_Y.sdg(QR_abc[0])
        Legitimate_parties_1_Y.y(self.QR[0])
        Legitimate_parties_1_Y.measure(self.QR[0],self.CR[0])
        
        # Legitimate_parties_1's qubit onto the R basis R = Rz(-pi/4)H = (X+Y)/sqrt(2)
        Legitimate_parties_1_R = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_R.rz(-np.pi/4,self.QR[0])
        Legitimate_parties_1_R.h(self.QR[0])
        Legitimate_parties_1_R.measure(self.QR[0],self.CR[0])
    
        # Legitimate_parties_1's qubit onto the V basis V = Rz(pi/4)H = (X-Y)/sqrt(2)
        Legitimate_parties_1_V = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_V.rz(np.pi/4,self.QR[0])
        Legitimate_parties_1_V.h(self.QR[0])
        Legitimate_parties_1_V.measure(self.QR[0],self.CR[0])
        Legitimate_parties_1_meas_circs = [Legitimate_parties_1_X, Legitimate_parties_1_Y, 
                                           Legitimate_parties_1_V, Legitimate_parties_1_R]
        Legitimate_parties_k_meas_circs = []
        Legitimate_parties_k_meas_circs2 = []
        for k in range(1,nq):
            # Legitimate_parties_k_basis = {'X','Y'}
            # Legitimate_parties_k's qubit onto the Y basis
            
            Legitimate_parties_k_Y = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_Y.sdg(self.QR[k])
            Legitimate_parties_k_Y.h(self.QR[k])
            Legitimate_parties_k_Y.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_Y2 = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_Y2.y(self.QR[k])
            Legitimate_parties_k_Y2.measure(self.QR[k],self.CR[k])
        
            # Legitimate_parties_k's qubit onto the X basis
            Legitimate_parties_k_X = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_X.h(self.QR[k])
            Legitimate_parties_k_X.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_X2 = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_X2.x(self.QR[k])
            Legitimate_parties_k_X2.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_meas_circs.append([Legitimate_parties_k_X,Legitimate_parties_k_Y])
            Legitimate_parties_k_meas_circs2.append([Legitimate_parties_k_X2,Legitimate_parties_k_Y2])
        
        Legitimate_parties_meas_circs = [Legitimate_parties_1_meas_circs]+Legitimate_parties_k_meas_circs
        Legitimate_parties_meas_circs2 = [Legitimate_parties_1_meas_circs]+Legitimate_parties_k_meas_circs2
        return Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2
        
# Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2 = legitimate_users_mes_circs(nq,N,QR,CR,CR_e)

    def circs(self,nq,N):
        Psi = self.N_particles_state(nq)
        Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2 = self.legitimate_users_mes_circs(nq,N)
        basis_choices = self.basis(nq,N)
        circuits = [] 
        circuits2 = []
        for k in range(N):
            if basis_choices[0][k] == 'X':
                circ = Psi.compose(Legitimate_parties_meas_circs[0][0])
                circ2 = Psi.compose(Legitimate_parties_meas_circs2[0][0])
                circuits.append(circ)
                circuits2.append(circ2)
            elif basis_choices[0][k] == 'Y':
                circ = Psi.compose(Legitimate_parties_meas_circs[0][1])
                circ2 = Psi.compose(Legitimate_parties_meas_circs2[0][1])
                circuits.append(circ)
                circuits2.append(circ2)
            elif basis_choices[0][k] == 'V':
                circ = Psi.compose(Legitimate_parties_meas_circs[0][2])
                circ2 = Psi.compose(Legitimate_parties_meas_circs2[0][2])
                circuits.append(circ)
                circuits2.append(circ2)
        
            elif basis_choices[0][k] == 'R':
                circ = Psi.compose(Legitimate_parties_meas_circs[0][3])
                circ2 = Psi.compose(Legitimate_parties_meas_circs2[0][3])
                circuits.append(circ)
                circuits2.append(circ2)
        
        for j in range(1,nq):
            for k in range(N):  
                circs = circuits[k]
                circs2 = circuits2[k]
                if basis_choices[j][k] == 'X':
                    circ = circs.compose(Legitimate_parties_meas_circs[j][0])
                    circ2 = circs2.compose(Legitimate_parties_meas_circs2[j][0])
                    circuits[k] = circ
                    circuits2[k] = circ2
                elif basis_choices[j][k] == 'Y':
                    circ = circs.compose(Legitimate_parties_meas_circs[j][1])
                    circ2 = circs2.compose(Legitimate_parties_meas_circs2[j][1])
                    circuits[k] = circ
                    circuits2[k] = circ2
        return circuits,circuits2
# circuits, circuits2 = circs(nq,N,basis_choices,Psi,Legitimate_parties_meas_circs,
#                  Legitimate_parties_meas_circs2)
    def simulation(self,nq,N,backend):
        circuits,circuits2 = self.circs(nq,N)
        Result = []
        Result2 = []
        
        pmv = generate_preset_pass_manager(
            optimization_level = 1, 
            backend = backend, 
            seed_transpiler = 1234,
            )
        for k in range(N):
            qc = circuits[k]
            qc2 = circuits2[k]
            qct = pmv.run(qc)
            qct2 = pmv.run(qc2)
            result = backend.run(qct, shots = 1).result() 
            result2 = backend.run(qct2, shots = 1).result() 
            res = result.get_counts()
            res2 = result2.get_counts()
            Result.append(res)
            Result2.append(res2)
        return Result,Result2

    def Legitimate_parties_secret_keys(self,nq,N):
        generated_keys = []
        Result = self.simulation(nq,N,backend)[1]
        basis_choices = self.basis(nq,N)
        keyss = np.array([[0]*N]*nq)
        for k in range(N):
            key = list(Result[k].keys())
            for j in range(nq):
                if key[0][nq+j+1] == '1':
                    keyss[j,k] = '1'
                elif key[0][nq+j+1] == '0':
                    keyss[j,k] = '0'
    
        Res = []
        L = [] #contains the set of measurement basis for key generation at potision determined by k_list
        k_list = [] #This is the list giving the position of all measurement basis where the first user
        # select either X or Y.
        for k in range(N):
            if basis_choices[0][k] == 'X':
                l = ['X']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])
                L.append(l)
                k_list.append(k)
            elif basis_choices[0][k] == 'Y':
                l = ['Y']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])            
                L.append(l)
                k_list.append(k)
            
        Y_counter = []#This contains the number of Y basis in each set of bases in L
        for k in range(len(L)):
            compt = 0
            for j in range(len(L[0])):
                if L[k][j] == 'Y':
                    compt += 1
            Y_counter.append(compt)
    
        Keys = [] #this contains the set of key bit of each user
        for k in range(nq):
            q = keyss[k]
            A = []
            for j in range(len(Y_counter)):
                if Y_counter[j]%2 == 0:# because when the is an odd number of Y basis selected the user
                                        # are most likely to get the same output results. 
                    A.append(str(q[k_list[j]]))  
            Keys.append(A)
        for k in range(len(Keys)):
            x = ''
            for j in Keys[k]:
                x += j
            generated_keys.append(x)
        
        return generated_keys
# generated_keys = Legitimate_parties_secret_keys(nq,N,Result2)
# generated_keys
    def KeyLength(self):
        generated_keys = self.Legitimate_parties_secret_keys(nq,N)
        return [len(generated_keys[k]) for k in range(len(generated_keys))]

    def Quantum_bit_error_rate(self,nq):
        Keylength = self.KeyLength()
        generated_keys = self.Legitimate_parties_secret_keys(nq,N)
        Numb_mismatching_bits = 0 
        x = generated_keys[0]
        if Keylength[0] == 0:
            print('Unsucessful')
        else:
            for j in range(Keylength[0]):
                for k in range(1,nq):
                    if x[j] != generated_keys[k][j]:
                        Numb_mismatching_bits +=1
                        break
            QBER = Numb_mismatching_bits/Keylength[0]*100
        return QBER


    def Corrrelation(self,nq,N):
        Corr_meas_basis = [] #contains the set of measurement basis for key generation at potision determined by k_list
        k_list = [] #This is the list giving the position of all measurement basis where the first user
        Meas_Results = []
        Result = self.simulation(nq,N,backend)[0]
        basis_choices = self.basis(nq,N)
        Psi = self.N_particles_state(nq)
        # select either X or Y.
        for k in range(N):
            if basis_choices[0][k] == 'V':
                l = 'V'
                for j in range(1,nq):
                    l += basis_choices[j][k]
                Corr_meas_basis.append(l)
                # k_list.append(k)
                Meas_Results.append(list(Result[k].keys()))
            elif basis_choices[0][k] == 'R':
                l = 'R'
                for j in range(1,nq):
                    l += basis_choices[j][k]            
                Corr_meas_basis.append(l)
                # k_list.append(k)
                Meas_Results.append(list(Result[k].keys()))
    
        Meas_Results_per_basis = {}#This dictionary contains as key the measurement bases and 
                                    #as values the list of the outcome measurements at the kth position
        for n, r in zip(Corr_meas_basis, Meas_Results):
            if n not in Meas_Results_per_basis:
                Meas_Results_per_basis[n] = []
            Meas_Results_per_basis[n].append(r)
    
        # the following lines is just to transform the list of lis of list of string into a list of list of string
        Meas_Results_per_basis_val = [[x[0] for x in list(Meas_Results_per_basis.values())[j]] 
                                      for j in range(len(list(Meas_Results_per_basis.values())))]
        Counts = []# this is a list of dictionaries that regroup the output results values 
        for k in range(len(Meas_Results_per_basis_val)):
            counts_mes = {}
            for item in set(Meas_Results_per_basis_val[k]):
                counts_mes[item] = Meas_Results_per_basis_val[k].count(item)
            Counts.append(counts_mes)
    
        Probs = [] # this is a list of list of probabilities where each list correspond to the probability
                    # of different outcomes for the respective measurement bases
        for j in range(len(Counts)):
            prob = []
            for k in range(len(list(Counts[j].values()))):
                n1 = list(Counts[j].values())[k]
                n2 = sum(Counts[j].values())
                prob.append(n1/n2)
            Probs.append(prob)
    
        expect_val = []#This list contains all the respective expectation values
        for j in range(len(Counts)):
            exp = 0
            for k in range(len(list(Counts[j].values()))):
                c_0 = 0
                for x in list(Counts[j].keys())[k][nq+1:]:
                    if x == '0':
                        c_0 += 1
                exp += (-1)**c_0*Probs[j][k]
            expect_val.append(exp)
    
        #The following block computes the correlation
        correlation = 0
        for k in range(len(expect_val)):
            compt = 0
            for j in range(len(list(Meas_Results_per_basis.keys())[0])):
                if list(Meas_Results_per_basis.keys())[k][j] == 'Y':
                    compt += 1
                elif list(Meas_Results_per_basis.keys())[k][j] == 'R':
                    compt += 1
            y = compt%4
            x = (-1)**(y*(y-1)/2)
            correlation += x*expect_val[k]
    
        # The following block compute the theoretical correlation
        coefs = list(Meas_Results_per_basis.values())
        ops_str = list(Meas_Results_per_basis.keys())
        for k in range(len(coefs)):
            compt = 0
            for j in range(len(ops_str[0])):
                if ops_str[k][j] == 'Y':
                    compt += 1
                elif ops_str[k][j] == 'R':
                    compt += 1
            y = compt%4
            x = (-1)**(y*(y-1)/2)
            coefs[k] = x#/coefs[k]
        # print(coefs)
        # print(ops_str)
        op_strings = []
        coeffs = []
        #v = x - y
        #R = x + y
        for k in range (len(ops_str)):
            if ops_str[k].startswith('V'):
                str1 = re.sub(r'[V]', 'X',ops_str[k])
                str2 = re.sub(r'[V]', 'Y',ops_str[k])
                op_strings.append(str1)
                op_strings.append(str2)
                coeffs.append(coefs[k]/np.sqrt(2))
                coeffs.append(-1*coefs[k]/np.sqrt(2))
            elif ops_str[k].startswith('R'):
                str1 = re.sub(r'[R]', 'X',ops_str[k])
                str2 = re.sub(r'[R]', 'Y',ops_str[k])
                op_strings.append(str1)
                op_strings.append(str2)
                coeffs.append(coefs[k]/np.sqrt(2))
                coeffs.append(coefs[k]/np.sqrt(2))
        observable = SparsePauliOp.from_list(list(zip(op_strings,coeffs)))
        estimator = Estimator()
        job = estimator.run(circuits = Psi, observables = observable)
        result = job.result()
        return correlation,float(result.values[0])

In [4]:
def main(nq,N,backend):
    qkd_net = QKD_Networks(nq,N,backend)
    return {"The number of users is ":nq,
            "The Quantum bit error rate is ":qkd_net.Quantum_bit_error_rate(nq),
            "The Correlation is ":qkd_net.Corrrelation(nq,N)[0],
            "The Theoretical Correlation is ":qkd_net.Corrrelation(nq,N)[1],
            "The Key length for all users is ":qkd_net.KeyLength(),
            "The secure key for all users is ":qkd_net.Legitimate_parties_secret_keys(nq,N),
           }
nq = 4  
N = 2000
main(nq,N,backend)  

C:\Users\alain\AppData\Local\Temp\ipykernel_11224\3208394848.py:346: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()
C:\Users\alain\AppData\Local\Temp\ipykernel_11224\3208394848.py:346: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()


{'The number of users is ': 4,
 'The Quantum bit error rate is ': 0.0,
 'The Correlation is ': 11.249084969524821,
 'The Theoretical Correlation is ': 11.313708498984758,
 'The Key length for all users is ': [506, 506, 506, 506],
 'The secure key for all users is ': ['10111100110010001000001001000101110010001100110001100010101101110100010110011110000100010101000010100001000000111000001100011001101010000011111000010011010110000100100111110111100000101001010010011100111110101011101001101110101101111100010000010111110010010000010100110010100100100110011000010111010101010000001110111100011010010001101001101111100111111110100110110010000110110101001101100100100011110010101000111000000010110101111001010010111100000101111010001010110001010000110011101101100110101000100111',
  '101111001100100010000010010001011100100011001100011000101011011101000101100111100001000101010000101000010000001110000011000110011010100000111110000100110101100001001001111101111000001010010100100111001111101010111010011

In [42]:
class QKD_Networks_with_Eavesdropper:
    def __init__(self, nq, N, backend):

        if isinstance(nq, int) == False:
            raise TypeError("The number of Users must be an integer greater than or equal to 2")
        else:  
            self.nq = nq
            self.N = N
            self.QR = QuantumRegister(nq)
            self.CR = ClassicalRegister(nq)
            self.QR_e = QuantumRegister(nq)
            self.CR_e = ClassicalRegister(nq) 

    def basis(self,nq,N):
        random.seed(17555)
        basis_1 = ['X','V','Y','R']
        basis_k = ['X','Y']
        basis_choices = [[random.choice(basis_1) for _ in range(N)]]+[[random.choice(basis_k) 
                                                    for _ in range(N)] for _ in range(nq-1) ]
        return basis_choices
    # basis_choices = basis(nq,N)

    def N_particles_state(self,nq):
        qc = QuantumCircuit(self.QR, self.CR, self.CR_e )
        qc.h(self.QR[0])
        for k in range(1,nq):
            qc.cx(self.QR[0],self.QR[k])
        return qc

    
    def legitimate_users_mes_circs(self,nq,N):
        #  Legitimate_parties_1's qubit onto the X basis
        Legitimate_parties_1_X = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_X.x(self.QR[0])
        Legitimate_parties_1_X.measure(self.QR[0],self.CR[0])
        
        # Legitimate_parties_1's qubit onto the Y basis 
        Legitimate_parties_1_Y = QuantumCircuit(self.QR,self.CR,self.CR_e)
        # Legitimate_parties_1_Y.sdg(QR_abc[0])
        Legitimate_parties_1_Y.y(self.QR[0])
        Legitimate_parties_1_Y.measure(self.QR[0],self.CR[0])
        
        # Legitimate_parties_1's qubit onto the R basis R = Rz(-pi/4)H = (X+Y)/sqrt(2)
        Legitimate_parties_1_R = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_R.rz(-np.pi/4,self.QR[0])
        Legitimate_parties_1_R.h(self.QR[0])
        Legitimate_parties_1_R.measure(self.QR[0],self.CR[0])
    
        # Legitimate_parties_1's qubit onto the V basis V = Rz(pi/4)H = (X-Y)/sqrt(2)
        Legitimate_parties_1_V = QuantumCircuit(self.QR,self.CR,self.CR_e)
        Legitimate_parties_1_V.rz(np.pi/4,self.QR[0])
        Legitimate_parties_1_V.h(self.QR[0])
        Legitimate_parties_1_V.measure(self.QR[0],self.CR[0])
        Legitimate_parties_1_meas_circs = [Legitimate_parties_1_X, Legitimate_parties_1_Y, 
                                           Legitimate_parties_1_V, Legitimate_parties_1_R]
        Legitimate_parties_k_meas_circs = []
        Legitimate_parties_k_meas_circs2 = []
        for k in range(1,nq):
            # Legitimate_parties_k_basis = {'X','Y'}
            # Legitimate_parties_k's qubit onto the Y basis
            
            Legitimate_parties_k_Y = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_Y.sdg(self.QR[k])
            Legitimate_parties_k_Y.h(self.QR[k])
            Legitimate_parties_k_Y.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_Y2 = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_Y2.y(self.QR[k])
            Legitimate_parties_k_Y2.measure(self.QR[k],self.CR[k])
        
            # Legitimate_parties_k's qubit onto the X basis
            Legitimate_parties_k_X = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_X.h(self.QR[k])
            Legitimate_parties_k_X.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_X2 = QuantumCircuit(self.QR,self.CR,self.CR_e)
            Legitimate_parties_k_X2.x(self.QR[k])
            Legitimate_parties_k_X2.measure(self.QR[k],self.CR[k])
            
            Legitimate_parties_k_meas_circs.append([Legitimate_parties_k_X,Legitimate_parties_k_Y])
            Legitimate_parties_k_meas_circs2.append([Legitimate_parties_k_X2,Legitimate_parties_k_Y2])
        
        Legitimate_parties_meas_circs = [Legitimate_parties_1_meas_circs]+Legitimate_parties_k_meas_circs
        Legitimate_parties_meas_circs2 = [Legitimate_parties_1_meas_circs]+Legitimate_parties_k_meas_circs2
        return Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2
        
# Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2 = legitimate_users_mes_circs(nq,N,QR,CR,CR_e)

    def Eve_basis(self,N):
        random.seed(57551)
        Eve_basis_choices = []# {X,Y}
        for j in range(N):      
            if random.uniform(0, 1) <= 0.5: # Eve uniformly generates X and Y with equal probabilities
                Eve_basis_choices.append('X')
            else: 
                Eve_basis_choices.append('Y')
        return Eve_basis_choices

    def eavesdropper_meas_circs(self,nq):
        Eve_k_meas_circs = []
        Eve_k_meas_circs2 = []
        for k in range(0,nq):
            # Eavedropper measures qubit in X basis
            Eve_X = QuantumCircuit(self.QR_e,self.CR,self.CR_e)
            Eve_X.h(self.QR_e[k])
            Eve_X.measure(self.QR_e[k],self.CR_e[k])
            
            # Eve measures qubit in X basis
            Eve_X2 = QuantumCircuit(self.QR_e,self.CR,self.CR_e)
            Eve_X2.x(self.QR_e[k])
            Eve_X2.measure(self.QR_e[k],self.CR_e[k])
            
            # Eve measures qubit in Y basis
            Eve_Y = QuantumCircuit(self.QR_e,self.CR,self.CR_e)
            Eve_Y.sdg(self.QR_e[k])
            Eve_Y.h(self.QR_e[k])
            Eve_Y.measure(self.QR_e[k],self.CR_e[k])
    
            # Eve measures qubit in Y basis
            Eve_Y2 = QuantumCircuit(self.QR_e,self.CR,self.CR_e)
            Eve_Y2.y(self.QR_e[k])
            Eve_Y2.measure(self.QR_e[k],self.CR_e[k])
            Eve_k_meas_circs.append([Eve_X,Eve_Y])
            Eve_k_meas_circs2.append([Eve_X2,Eve_Y2])
        return Eve_k_meas_circs,Eve_k_meas_circs2

    def circs_with_eavesdropper(self,nq,N):
        Eve_k_meas_circs,Eve_k_meas_circs2 = self.eavesdropper_meas_circs(nq)
        Eve_basis_choices = self.Eve_basis(N)
        basis_choices = self.basis(nq,N)
        Psi = self.N_particles_state(nq)
        Legitimate_parties_meas_circs,Legitimate_parties_meas_circs2 = self.legitimate_users_mes_circs(nq,N)
        circuits_Eve = [] 
        circuits_Eve2 = [] 
        #basis_choices[[],[]...[]]
        
        for k in range(N):
            if basis_choices[0][k] == 'X':
                if Eve_basis_choices[k] == 'X':
                    circ = Psi & Eve_k_meas_circs[0][0] & Legitimate_parties_meas_circs[0][0]
                    circ2 = Psi & Eve_k_meas_circs2[0][0] & Legitimate_parties_meas_circs2[0][0]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
                elif Eve_basis_choices[k] == 'Y':
                    circ = Psi & Eve_k_meas_circs[0][1] & Legitimate_parties_meas_circs[0][0]
                    circ2 = Psi & Eve_k_meas_circs2[0][1] & Legitimate_parties_meas_circs2[0][0]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
            elif basis_choices[0][k] == 'Y':
                if Eve_basis_choices[k] == 'X':
                    circ = Psi & Eve_k_meas_circs[0][0] & Legitimate_parties_meas_circs[0][1]
                    circ2 = Psi & Eve_k_meas_circs2[0][0]& Legitimate_parties_meas_circs2[0][1]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
                elif Eve_basis_choices[k] == 'Y':
                    circ = Psi & Eve_k_meas_circs[0][1] & Legitimate_parties_meas_circs[0][1]
                    circ2 = Psi & Eve_k_meas_circs2[0][1] & Legitimate_parties_meas_circs2[0][1]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
            elif basis_choices[0][k] == 'V':
                if Eve_basis_choices[k] == 'X':
                    circ = Psi & Eve_k_meas_circs[0][0] & Legitimate_parties_meas_circs[0][2]
                    circ2 = Psi & Eve_k_meas_circs2[0][0] & Legitimate_parties_meas_circs2[0][2]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
                elif Eve_basis_choices[k] == 'Y':
                    circ = Psi & Eve_k_meas_circs[0][1] & Legitimate_parties_meas_circs[0][2]
                    circ2 = Psi & Eve_k_meas_circs2[0][1] & Legitimate_parties_meas_circs2[0][2]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
        
            elif basis_choices[0][k] == 'R':
                if Eve_basis_choices[k] == 'X':
                    circ = Psi & Eve_k_meas_circs[0][0] & Legitimate_parties_meas_circs[0][3]
                    circ2 = Psi & Eve_k_meas_circs2[0][0] & Legitimate_parties_meas_circs2[0][3]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
                elif Eve_basis_choices[k] == 'Y':
                    circ = Psi & Eve_k_meas_circs[0][1] & Legitimate_parties_meas_circs[0][3]
                    circ2 = Psi & Eve_k_meas_circs2[0][1] & Legitimate_parties_meas_circs2[0][3]
                    circuits_Eve.append(circ)
                    circuits_Eve2.append(circ2)
        for j in range(1,nq):
            for k in range(N):  
                circs = circuits_Eve[k]
                circs2 = circuits_Eve2[k]
                if basis_choices[j][k] == 'X':
                    if Eve_basis_choices[k] == 'X':
                        circs = circs & Eve_k_meas_circs[j][0] & Legitimate_parties_meas_circs[j][0]
                        circs2 = circs2 & Eve_k_meas_circs2[j][0] & Legitimate_parties_meas_circs2[j][0]
                        circuits_Eve[k] = circs
                        circuits_Eve2[k] = circs2
                    elif Eve_basis_choices[k] == 'Y':
                        circs = circs & Eve_k_meas_circs[j][1] & Legitimate_parties_meas_circs[j][0]
                        circs2 = circs2 & Eve_k_meas_circs2[j][1] & Legitimate_parties_meas_circs2[j][0]
                        circuits_Eve[k] = circs
                        circuits_Eve2[k] = circs2
                elif basis_choices[j][k] == 'Y':
                    if Eve_basis_choices[k] == 'X':
                        circs = circs & Eve_k_meas_circs[j][0] & Legitimate_parties_meas_circs[j][1]
                        circs2 = circs2 &Eve_k_meas_circs2[j][0] & Legitimate_parties_meas_circs2[j][1]
                        circuits_Eve[k] = circs
                        circuits_Eve2[k] = circs2
                    elif Eve_basis_choices[k] == 'Y':
                        circs = circs & Eve_k_meas_circs[j][1] & Legitimate_parties_meas_circs[j][1]
                        circs2 = circs2 & Eve_k_meas_circs2[j][1] & Legitimate_parties_meas_circs2[j][1]
                        circuits_Eve[k] = circs
                        circuits_Eve2[k] = circs2
        return circuits_Eve,circuits_Eve2
    
    def simulations(self,nq,N,backend):
        circuits_Eve,circuits_Eve2 = self.circs_with_eavesdropper(nq,N)
        Result_Eve = []
        Result_Eve2 = []
        
        pmv = generate_preset_pass_manager(
            optimization_level = 1, 
            backend=backend, 
            seed_transpiler = 1234,
        )
        for k in range(N):
            qc = circuits_Eve[k]
            qc2 = circuits_Eve2[k]
            result = backend.run(pmv.run(qc), shots = 1).result() 
            result2 = backend.run(pmv.run(qc2), shots = 1).result()
        
            res = result.get_counts()
            res2 = result2.get_counts()
        
            Result_Eve.append(res)
            Result_Eve2.append(res2)
        return Result_Eve,Result_Eve2
# Result_Eve,Result_Eve2= simulations(N,backend,circuits_Eve,circuits_Eve2)

    def Legitimate_parties_secret_key_with_Eve(self,nq,N):
        Result_Eve = self.simulations(nq,N,backend)[1]
        basis_choices =  self.basis(nq,N)
        keyss_Legitimate_parties = np.array([[0]*N]*nq)
        for k in range(N):
            key = list(Result_Eve[k].keys())
            for j in range(nq):
                if key[0][nq+j+1] == '1':
                    keyss_Legitimate_parties[j,k] = '1'
                elif key[0][nq+j+1] == '0':
                    keyss_Legitimate_parties[j,k] = '0'
        
        Res = []
        L = [] #contains the set of measurement basis for key generation at potision determined by k_list
        k_list = [] #This is the list giving the position of all measurement basis where the first user
        # select either X or Y.
        for k in range(N):
            if basis_choices[0][k] == 'X':
                l = ['X']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])
                L.append(l)
                k_list.append(k)
            elif basis_choices[0][k] == 'Y':
                l = ['Y']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])            
                L.append(l)
                k_list.append(k)
    
        Y_counter = []#This contains the number of Y basis in each set of bases in L
        for k in range(len(L)):
            compt = 0
            for j in range(len(L[0])):
                if L[k][j] == 'Y':
                    compt += 1
            Y_counter.append(compt)
        
        Keys_Legitimate_parties = [] #this contains the set of key bit of each user
        for k in range(nq):
            q = keyss_Legitimate_parties[k]
            A = []
            for j in range(len(Y_counter)):
                if Y_counter[j]%2 == 0:# because when the is an odd number of Y basis selected the user
                                        # are most likely to get the same output results. 
                    A.append(str(q[k_list[j]]))  
            Keys_Legitimate_parties.append(A)
        generated_keys_Legitimate_parties = []
        for k in range(len(Keys_Legitimate_parties)):
            x = ''
            for j in Keys_Legitimate_parties[k]:
                x += j
            generated_keys_Legitimate_parties.append(x)
            
        return generated_keys_Legitimate_parties
# generated_keys_Legitimate_parties = Legitimate_parties_secret_key_with_Eve(nq,N,Result_Eve2)
# generated_keys_Legitimate_parties

    def keylength(self):
        generated_keys_Legitimate_parties = self.Legitimate_parties_secret_key_with_Eve(nq,N)
        return [len(generated_keys_Legitimate_parties[k]) 
             for k in range(len(generated_keys_Legitimate_parties))]


    def Legitimate_parties_QBER_with_eve(self,nq,N):
        Keylength = self.keylength()
        keys = self.Legitimate_parties_secret_key_with_Eve(nq,N)
        Numb_mismatching_bits = 0 
        x = keys[0]
        if Keylength[0] == 0:
            print('Unsucessful')
        else:
            for j in range(Keylength[0]):
                for k in range(1,nq):
                    if x[j] != keys[k][j]:
                        Numb_mismatching_bits +=1
                        break
            QBER = Numb_mismatching_bits/Keylength[0]*100
        return QBER


    def Eve_extracted_keys(self,nq,N):
        Result_Eve = self.simulations(nq,N,backend)[1]
        basis_choices =  self.basis(nq,N)
        keyss_Eve = np.array([[0]*N]*nq)
        for k in range(N):
            key = list(Result_Eve[k].keys())
            for j in range(nq):
                if key[0][j] == '1':
                    keyss_Eve[j,k] = '1'
                elif key[0][j] == '0':
                    keyss_Eve[j,k] = '0'
    
        
        L = [] #contains the set of measurement basis for key generation at potision determined by k_list
        k_list = [] #This is the list giving the position of all measurement basis where the first user
        # select either X or Y.
        for k in range(N):
            if basis_choices[0][k] == 'X':
                l = ['X']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])
                L.append(l)
                k_list.append(k)
            elif basis_choices[0][k] == 'Y':
                l = ['Y']
                for j in range(1,nq):
                    l.append(basis_choices[j][k])            
                L.append(l)
                k_list.append(k)
    
        Y_counter = []#This contains the number of Y basis in each set of bases in L
        for k in range(len(L)):
            compt = 0
            for j in range(len(L[0])):
                if L[k][j] == 'Y':
                    compt += 1
            Y_counter.append(compt)
        
        #Here I am extracting the bit where the measurement basis correspond to the 
        Keys_Eve = [] #this contains the set of key bit of each user
        for k in range(nq):
            q = keyss_Eve[k]
            A = []
            for j in range(len(Y_counter)):
                if Y_counter[j]%2 == 0:# because when the is an odd number of Y basis selected the user
                                        # are most likely to get the same output results. 
                    A.append(str(q[k_list[j]]))  
            Keys_Eve.append(A)
        generated_keys_Eve = []
        for k in range(len(Keys_Eve)):
            x = ''
            for j in Keys_Eve[k]:
                x += j
            generated_keys_Eve.append(x)
    
        return generated_keys_Eve

    def Percentage_info_learnt_by_Eve(self,nq,N):
        generated_keys_Eve = self.Eve_extracted_keys(nq,N)
        generated_keys = self.Legitimate_parties_secret_key_with_Eve(nq,N)
        Keylength = self.keylength()
        Percentage_Eve_mismatching_bits = [] 
        for k in range(len(generated_keys_Eve)):
            Numb_mismatching_bits = 0
            x = generated_keys_Eve[k]
            Eve_Keylength = len(generated_keys_Eve[k])
            if Eve_Keylength == 0:
                Numb_Eve_mismatching_bits.append('Unsucessful')
            else:
                for j in range(Eve_Keylength):
                    if x[j] != generated_keys[k][j]:
                        Numb_mismatching_bits +=1
                        # break
                qber = Numb_mismatching_bits/Keylength[0]*100
                Percentage_Eve_mismatching_bits.append(qber)
        return Percentage_Eve_mismatching_bits

    def Corrrelation(self,nq,N):
        Corr_meas_basis = [] #contains the set of measurement basis for key generation at potision determined by k_list
        k_list = [] #This is the list giving the position of all measurement basis where the first user
        Meas_Results = []
        Result = self.simulations(nq,N,backend)[0]
        basis_choices = self.basis(nq,N)
        Psi = self.N_particles_state(nq)
        # select either X or Y.
        for k in range(N):
            if basis_choices[0][k] == 'V':
                l = 'V'
                for j in range(1,nq):
                    l += basis_choices[j][k]
                Corr_meas_basis.append(l)
                # k_list.append(k)
                Meas_Results.append(list(Result[k].keys()))
            elif basis_choices[0][k] == 'R':
                l = 'R'
                for j in range(1,nq):
                    l += basis_choices[j][k]            
                Corr_meas_basis.append(l)
                # k_list.append(k)
                Meas_Results.append(list(Result[k].keys()))
    
        Meas_Results_per_basis = {}#This dictionary contains as key the measurement bases and 
                                    #as values the list of the outcome measurements at the kth position
        for n, r in zip(Corr_meas_basis, Meas_Results):
            if n not in Meas_Results_per_basis:
                Meas_Results_per_basis[n] = []
            Meas_Results_per_basis[n].append(r)
    
        # the following lines is just to transform the list of lis of list of string into a list of list of string
        Meas_Results_per_basis_val = [[x[0] for x in list(Meas_Results_per_basis.values())[j]] 
                                      for j in range(len(list(Meas_Results_per_basis.values())))]
        Counts = []# this is a list of dictionaries that regroup the output results values 
        for k in range(len(Meas_Results_per_basis_val)):
            counts_mes = {}
            for item in set(Meas_Results_per_basis_val[k]):
                counts_mes[item] = Meas_Results_per_basis_val[k].count(item)
            Counts.append(counts_mes)
    
        Probs = [] # this is a list of list of probabilities where each list correspond to the probability
                    # of different outcomes for the respective measurement bases
        for j in range(len(Counts)):
            prob = []
            for k in range(len(list(Counts[j].values()))):
                n1 = list(Counts[j].values())[k]
                n2 = sum(Counts[j].values())
                prob.append(n1/n2)
            Probs.append(prob)
    
        expect_val = []#This list contains all the respective expectation values
        for j in range(len(Counts)):
            exp = 0
            for k in range(len(list(Counts[j].values()))):
                c_0 = 0
                for x in list(Counts[j].keys())[k][nq+1:]:
                    if x == '0':
                        c_0 += 1
                exp += (-1)**c_0*Probs[j][k]
            expect_val.append(exp)
    
        #The following block computes the correlation
        correlation = 0
        for k in range(len(expect_val)):
            compt = 0
            for j in range(len(list(Meas_Results_per_basis.keys())[0])):
                if list(Meas_Results_per_basis.keys())[k][j] == 'Y':
                    compt += 1
                elif list(Meas_Results_per_basis.keys())[k][j] == 'R':
                    compt += 1
            y = compt%4
            x = (-1)**(y*(y-1)/2)
            correlation += x*expect_val[k]
    
        # The following block compute the theoretical correlation
        coefs = list(Meas_Results_per_basis.values())
        ops_str = list(Meas_Results_per_basis.keys())
        for k in range(len(coefs)):
            compt = 0
            for j in range(len(ops_str[0])):
                if ops_str[k][j] == 'Y':
                    compt += 1
                elif ops_str[k][j] == 'R':
                    compt += 1
            y = compt%4
            x = (-1)**(y*(y-1)/2)
            coefs[k] = x#/coefs[k]
        # print(coefs)
        # print(ops_str)
        op_strings = []
        coeffs = []
        #v = x - y
        #R = x + y
        for k in range (len(ops_str)):
            if ops_str[k].startswith('V'):
                str1 = re.sub(r'[V]', 'X',ops_str[k])
                str2 = re.sub(r'[V]', 'Y',ops_str[k])
                op_strings.append(str1)
                op_strings.append(str2)
                coeffs.append(coefs[k]/np.sqrt(2))
                coeffs.append(-1*coefs[k]/np.sqrt(2))
            elif ops_str[k].startswith('R'):
                str1 = re.sub(r'[R]', 'X',ops_str[k])
                str2 = re.sub(r'[R]', 'Y',ops_str[k])
                op_strings.append(str1)
                op_strings.append(str2)
                coeffs.append(coefs[k]/np.sqrt(2))
                coeffs.append(coefs[k]/np.sqrt(2))
        observable = SparsePauliOp.from_list(list(zip(op_strings,coeffs)))
        estimator = Estimator()
        job = estimator.run(circuits = Psi, observables = observable)
        result = job.result()
        return correlation,float(result.values[0])

In [44]:

def main_with_eavesdropper(nq,N,backend):
    qkd_net_Eve = QKD_Networks_with_Eavesdropper(nq, N, backend)
    return {"The number of users is ":nq,
            "The Quantum bit error rate is ":qkd_net_Eve.Legitimate_parties_QBER_with_eve(nq,N),
            "The Correlation is ":qkd_net_Eve.Corrrelation(nq,N)[0],
            "The Theoretical Correlation is ":qkd_net_Eve.Corrrelation(nq,N)[1],
            "Percentage of Eve mismatching bits":qkd_net_Eve.Percentage_info_learnt_by_Eve(nq,N),
            "The Key length for all users is ":qkd_net_Eve.keylength(),
            "The secure key for all users is ":qkd_net_Eve.Legitimate_parties_secret_key_with_Eve(nq,N),
           }
nq = 4  
N = 2000
main_with_eavesdropper(nq,N,backend) 

C:\Users\alain\AppData\Local\Temp\ipykernel_11224\2532630026.py:503: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()
C:\Users\alain\AppData\Local\Temp\ipykernel_11224\2532630026.py:503: DeprecationWarning: The class ``qiskit.primitives.estimator.Estimator`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseEstimatorV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `Estimator` class is `StatevectorEstimator`.
  estimator = Estimator()


{'The number of users is ': 4,
 'The Quantum bit error rate is ': 0.0,
 'The Correlation is ': 0.2808734215838185,
 'The Theoretical Correlation is ': 11.313708498984758,
 'Percentage of Eve mismatching bits': [48.418972332015805,
  48.418972332015805,
  48.418972332015805,
  48.418972332015805],
 'The Key length for all users is ': [506, 506, 506, 506],
 'The secure key for all users is ': ['00101000011010010001001011100101100101101001011010001000101100101010110101110110000011010110101100011001000000101001111001001010001111010101101001001000010010100011110100001111101011001110111000011111001110010011011000000010101001100100000000010101100111001000001001001011010110000010110100100001100011000110100100011110100111100010000010111101110100100100110000001000011110011011011100011011110101100000001011111111011010010100101110110001100100001000110101110000100110001111101110111000101010010001101100',
  '00101000011010010001001011100101100101101001011010001000101100101010110101110110000011010110